### Data Cleaning, Preprocessing, and EDA.
###### Data Source (UCI credit card default) https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

- Fixes invalid categories
- Keeps PAY variables untouched despite invalid values in PAY_0,...,PAY_6. Removing data may result in a significant reduction in observations and a suitable imputation method was not identified.
- Encodes categorical variables properly
- Data preparation for: logistic regression, ridge/lasso

In [7]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Data Cleaning

In [8]:
# load data
default_of_credit_card_clients = fetch_ucirepo(id=350)

X = default_of_credit_card_clients.data.features.copy()
y = default_of_credit_card_clients.data.targets.copy()

# If y is a one-column DataFrame, convert to a Series
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target unique values:", sorted(y.unique()))

X shape: (30000, 23)
y shape: (30000,)
Target unique values: [np.int64(0), np.int64(1)]


In [9]:
# rename columns
rename_map = {
    "X1": "LIMIT_BAL",
    "X2": "SEX",
    "X3": "EDUCATION",
    "X4": "MARRIAGE",
    "X5": "AGE",
    "X6": "PAY_0",
    "X7": "PAY_2",
    "X8": "PAY_3",
    "X9": "PAY_4",
    "X10": "PAY_5",
    "X11": "PAY_6",
    "X12": "BILL_AMT1",
    "X13": "BILL_AMT2",
    "X14": "BILL_AMT3",
    "X15": "BILL_AMT4",
    "X16": "BILL_AMT5",
    "X17": "BILL_AMT6",
    "X18": "PAY_AMT1",
    "X19": "PAY_AMT2",
    "X20": "PAY_AMT3",
    "X21": "PAY_AMT4",
    "X22": "PAY_AMT5",
    "X23": "PAY_AMT6",
}

X = X.rename(columns=rename_map)
y = y.rename("DEFAULT")

print("\nRenamed columns:")
print(X.columns.tolist())


Renamed columns:
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


In [10]:
# ensure validity of data value and data type
# -----------------------------
# EDUCATION: collapse undocumented categories
X["EDUCATION"] = X["EDUCATION"].replace({0: 4, 5: 4, 6: 4})

# MARRIAGE: collapse undocumented category
X["MARRIAGE"] = X["MARRIAGE"].replace({0: 3})

# Convert to categorical
categorical_cols = ["SEX", "EDUCATION", "MARRIAGE"]

for col in categorical_cols:
    X[col] = X[col].astype("category")

# -----------------------------
# 5. PAY variables (leave as-is)
# -----------------------------
pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

# Ensure numeric (no transformation)
X[pay_cols] = X[pay_cols].astype(int)

In [11]:
# define variable groups
financial_cols = [
    "LIMIT_BAL",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
]

behavioral_cols = pay_cols

demographic_cols = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]

In [12]:
# train-test split (5 fold cross validation)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
# preprocessing for models

# Separate numeric vs categorical
numeric_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
    ]
)

# Fit-transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# -----------------------------
# 9. Sanity checks
# -----------------------------
print("Shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nDefault rate:")
print("Train:", y_train.mean())
print("Test:", y_test.mean())

print("\nProcessed shapes:")
print(X_train_processed.shape)
print(X_test_processed.shape)

Shapes:
X_train: (24000, 23)
X_test: (6000, 23)

Default rate:
Train: 0.22120833333333334
Test: 0.22116666666666668

Processed shapes:
(24000, 26)
(6000, 26)


In [14]:
X

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,20000,2,2,1,24,2,2,-1,-1,-2,...,689,0,0,0,0,689,0,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,...,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,90000,2,2,2,34,0,0,0,0,0,...,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,50000,2,2,1,37,0,0,0,0,0,...,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,50000,1,2,1,57,-1,0,-1,0,0,...,35835,20940,19146,19131,2000,36681,10000,9000,689,679
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,1,39,0,0,0,0,0,...,208365,88004,31237,15980,8500,20000,5003,3047,5000,1000
29996,150000,1,3,2,43,-1,-1,-1,-1,0,...,3502,8979,5190,0,1837,3526,8998,129,0,0
29997,30000,1,2,2,37,4,3,2,-1,0,...,2758,20878,20582,19357,0,0,22000,4200,2000,3100
29998,80000,1,3,1,41,1,-1,0,0,0,...,76304,52774,11855,48944,85900,3409,1178,1926,52964,1804


In [16]:
y

0        1
1        1
2        0
3        0
4        0
        ..
29995    0
29996    0
29997    1
29998    1
29999    1
Name: DEFAULT, Length: 30000, dtype: int64